# Resolver call A: contract draft

Tests `llm.generate()` with `ContractDraft` (OpenRouter chat + strict JSON schema).

- Needs `LLM_API_KEY` in `.env` at the repo root.
- Set `DEMURRAGE_JEV_CACHE=1` to cache responses under `.jev_cache/`.
- Optional: `DEMURRAGE_RESOLVER_MODEL` (default `openai/gpt-4.1-mini`).

In [ ]:
import os
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
os.environ.setdefault("DEMURRAGE_JEV_CACHE", "1")

from demurrage import llm
from demurrage.emails import parse_eml_file
from demurrage.resolver_schemas import ContractDraft

print("resolver model:", llm.resolver_model_name())
print("cache:", llm.cache_mode(), "→", llm.cache_dir())

In [ ]:
CONTRACT_EMAILS = [
    "emails/email_01.eml",
    "emails/email_05.eml",
]

blocks = []
for path in CONTRACT_EMAILS:
    p = parse_eml_file(Path(path))
    blocks.append(
        f"--- {p.source_id} ---\n"
        f"From: {p.from_address}\nDate: {p.sent_at}\nSubject: {p.subject}\n\n"
        f"{p.body_text}"
    )
user = "\n\n".join(blocks)
print(user[:800], "...")

In [ ]:
SYSTEM = """You are the Halcyon Tankers voyage resolver (contract only).

Rules:
- The broker recap is the baseline contract; each addendum replaces only the term it amends.
- Quote verbatim from the emails for every material field (evidence.quote).
- Do not compute laytime hours, durations or demurrage amounts.
- If the email and DOMAIN.md disagree on a term, the email wins.
- Use status resolved only when allowance, turn time after NOR, rate and currency are known.
- Return ISO 8601 timestamps only when parsing event times (not needed for contract call).
"""

draft = llm.generate(SYSTEM, user, ContractDraft)
print(draft.model_dump_json(indent=2))

In [ ]:
print("Schema excerpt (laytime_allowance_hours description):")
schema = ContractDraft.model_json_schema()
print(schema["properties"]["laytime_allowance_hours"].get("description", "?")[:120])